In [ ]:
with prepared as (
    select
        client_id,
        campaign_name,
        mrc_start_date,
        date_trunc('month', mrc_start_date)::date as month_dt,

        case
            when lower(com_cus_sgr_desc) like '%200%' then 200
            when lower(com_cus_sgr_desc) like '%300%' then 300
            when lower(com_cus_sgr_desc) like '%400%' then 400
            when lower(com_cus_sgr_desc) like '%500%' then 500
            else null
        end as nominal_amount

    from cvm_sbx.YOUR_TABLE
    where client_id is not null
),

client_flight_history as (
    select
        client_id,
        campaign_name,
        mrc_start_date,
        month_dt,
        nominal_amount,

        lag(nominal_amount) over (
            partition by client_id
            order by mrc_start_date, campaign_name
        ) as prev_nominal,

        lag(mrc_start_date) over (
            partition by client_id
            order by mrc_start_date, campaign_name
        ) as prev_mrc_start_date

    from prepared
    where nominal_amount is not null
),

nominal_comparison as (
    select
        *,
        case
            when nominal_amount = prev_nominal then 1
            else 0
        end as same_nominal_flag
    from client_flight_history
    where prev_nominal is not null
)

select
    month_dt,
    count(*) as transitions_cnt,
    sum(same_nominal_flag) as same_nominal_cnt,
    round(sum(same_nominal_flag) * 100.0 / count(*), 2) as same_nominal_pct
from nominal_comparison
group by month_dt
order by month_dt;